#### Text-Only RAG 

This notebook demonstrates how to ingest a PDF slide deck, extract slide-level text, embed the summaries, and store them in Redis using LangChain, OpenAI, and a local Docker Redis instance.

In [ ]:
#!pip install langchain-redis

In [5]:
# Import Required Libraries
import redis

In [6]:
# Redis connection settings
REDIS_URL = "redis://localhost:6379"

List all modules in redis client

In [7]:
r = redis.Redis.from_url(REDIS_URL)

try:
    modules = r.execute_command("MODULE", "LIST")
    
    print("Loaded Redis modules:")
    for module in modules:
        print(f"- {module[1].decode()} (version {module[3]})")
except Exception as e:
    print("Could not retrieve module list:", e)

Loaded Redis modules:
- RedisCompat (version 1)
- ReJSON (version 20809)
- redisgears_2 (version 20020)
- bf (version 20816)
- timeseries (version 11206)
- search (version 21020)


Test the connection

In [8]:
# Test Redis server connection
try:
    
    r.ping()
    print("✅ Successfully connected to Redis at", REDIS_URL)
except Exception as e:
    print("❌ Redis connection failed:", e)

✅ Successfully connected to Redis at redis://localhost:6379


In [9]:
print("RediSearch Index Names:")
for idx in r.execute_command("FT._LIST"):
    print("-", idx.decode())

RediSearch Index Names:
- nvidia_investor_pdf_vectors


In [10]:
# Show Redis connection details and current database number
try:
    connection_kwargs = r.connection_pool.connection_kwargs
    
    print("Redis connection details:")
    print(f"  Host: {connection_kwargs.get('host', 'N/A')}")
    print(f"  Port: {connection_kwargs.get('port', 'N/A')}")
    print(f"  DB: {connection_kwargs.get('db', 0)}")
    print(f"  URL: {REDIS_URL}")
except Exception as e:
    print("Error retrieving connection details:", e)

Redis connection details:
  Host: localhost
  Port: 6379
  DB: 0
  URL: redis://localhost:6379


In [ ]:
#r.execute_command("FT.DROPINDEX", "my_vectors", "DD")  # "DD" also deletes the documents

In [ ]:
#r.execute_command("FT.DROPINDEX", "embeddings-index", "DD")

ResponseError: Unknown Index name

In [12]:
# Deleting Index Names

for idx in r.execute_command("FT._LIST"):
    r.execute_command("FT.DROPINDEX", idx.decode(), "DD")

In [13]:
r.execute_command("FT._LIST")

[]

#### Data load and split

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
pdf_path = r"D:\Makesh\Working\AI\RPS\Assignments\Capstone\cap-02\nvda-f3q24-investor-presentation-final.pdf"

# Load PDF
loader      = PyPDFLoader(pdf_path)
documents   = loader.load()

In [16]:
documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2023-12-05T11:13:13-08:00', 'title': 'PowerPoint Presentation', 'author': 'Sophie Nguyen', 'moddate': '2023-12-05T11:13:13-08:00', 'source': 'D:\\Makesh\\Working\\AI\\RPS\\Assignments\\Capstone\\cap-02\\nvda-f3q24-investor-presentation-final.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}, page_content='Investor Presentation \nQ3 FY24\nNovember 27, 2023'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2023-12-05T11:13:13-08:00', 'title': 'PowerPoint Presentation', 'author': 'Sophie Nguyen', 'moddate': '2023-12-05T11:13:13-08:00', 'source': 'D:\\Makesh\\Working\\AI\\RPS\\Assignments\\Capstone\\cap-02\\nvda-f3q24-investor-presentation-final.pdf', 'total_pages': 65, 'page': 1, 'page_label': '2'}, page_content="Except for the historical 

In [17]:
# Basic statistics
num_pages = len(documents)
all_text  = " ".join([doc.page_content for doc in documents])
num_chars = len(all_text)
num_words = len(all_text.split())

print(f"PDF Path: {pdf_path}")
print(f"Number of pages: {num_pages}")
print(f"Total characters: {num_chars}")
print(f"Total words: {num_words}")

PDF Path: D:\Makesh\Working\AI\RPS\Assignments\Capstone\cap-02\nvda-f3q24-investor-presentation-final.pdf
Number of pages: 65
Total characters: 51836
Total words: 7861


In [18]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")

# print("First chunk preview:")
# print(chunks[0].page_content[:500])

Number of chunks: 87


#### Embedding fn

In [19]:
# Define embedding function using LangChain and OpenAI
from langchain_openai import OpenAIEmbeddings

embedding_fn = OpenAIEmbeddings()

In [20]:
from langchain_redis import RedisVectorStore, RedisConfig

In [21]:
config = RedisConfig(
    index_name     = "nvidia_investor_pdf_vectors",
    redis_url      = REDIS_URL,
    distance_metric= "COSINE"  # Options: COSINE, L2, IP
)

# config = RedisConfig(
#     index_name="my_index",
#     redis_url="redis://localhost:6379",
#     distance_metric="COSINE",
#     key_prefix="my_prefix",
#     vector_datatype="FLOAT32",
#     storage_type="hash",
#     metadata_schema=[
#         {"name": "category", "type": "tag"},
#         {"name": "price", "type": "numeric"}
#     ]
# )

In [22]:
vector_store = RedisVectorStore(embedding_fn, config=config)

22:55:53 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [23]:
# Add chunks to Redis vector store
texts     = [chunk.page_content for chunk in chunks]

In [24]:
# Metadata: only page_number
metadatas = [
    {"page_number": chunk.metadata.get("page", None)}
    for chunk in chunks
]

In [25]:
vector_store.add_texts(texts, 
                       metadatas=metadatas)

print(f"Added {len(texts)} chunks to Redis vector store.")

22:56:04 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Added 87 chunks to Redis vector store.


#### Queries from the NVIDIA Investor Presentation PDF with Difficulty Analysis and Likely Page Numbers

1. **What are the key financial highlights and revenue growth drivers for NVIDIA in the most recent quarter?**  
*Difficulty:* Requires synthesizing quantitative data from multiple slides, understanding financial terminology, and distinguishing between highlights and detailed figures.  
*Likely pages:* 2, 3, 4, 5

2. **How does NVIDIA describe the impact of generative AI on its data center business and future outlook?**  
*Difficulty:* Involves extracting nuanced statements about technology trends, future projections, and business strategy, which may be spread across several slides and use non-obvious language.  
*Likely pages:* 4, 6, 7, 8

3. **Summarize the main advancements in NVIDIA’s GPU architecture as presented in the investor slides.**  
*Difficulty:* Demands technical comprehension, the ability to aggregate details from technical diagrams and text, and to distinguish new advancements from background information.  
*Likely pages:* 9, 10, 11

4. **What strategic partnerships or collaborations are highlighted, and how do they contribute to NVIDIA’s growth?**  
*Difficulty:* Requires identifying named partners, understanding the context of each partnership, and inferring their business impact, which may not be explicitly stated.  
*Likely pages:* 12, 13

5. **How is NVIDIA addressing the challenges and opportunities in the automotive sector, according to the presentation?**  
*Difficulty:* Involves finding sector-specific content, interpreting both challenges and opportunities, and understanding how NVIDIA’s solutions map to industry needs.  
*Likely pages:* 14, 15

6. **What are the main components and benefits of the NVIDIA Omniverse platform as described in the PDF?**  
*Difficulty:* Requires technical synthesis, mapping features to benefits, and possibly integrating information from multiple slides or diagrams.  
*Likely pages:* 16, 17

7. **How does NVIDIA position its role in the AI ecosystem, and what are its competitive advantages?**  
*Difficulty:* Involves interpreting marketing language, extracting implicit and explicit claims, and comparing to industry context, which may not be directly stated.  
*Likely pages:* 6, 8, 18

8. **What are the key risks and uncertainties mentioned in the investor presentation?**  
*Difficulty:* Risks may be scattered, phrased cautiously, or buried in footnotes or disclaimers, requiring careful reading and synthesis.  
*Likely pages:* 19, 20

9. **How does NVIDIA’s approach to sustainability and ESG (Environmental, Social, Governance) appear in the slides?**  
*Difficulty:* ESG content may be brief, high-level, or use non-standard terminology, requiring inference and synthesis from limited data.  
*Likely pages:* 21, 22

10. **What future product launches or technology roadmaps are discussed, and what is their expected impact on NVIDIA’s business?**  
*Difficulty:* Requires identifying forward-looking statements, understanding technical and business implications, and connecting roadmap items to business outcomes.  
*Likely pages:* 11, 18, 23

#### Query Selection and Input

Use the cell below to select a query from the list above or enter your own custom question about the NVIDIA investor presentation PDF.

In [ ]:
# Interactive query selection using ipywidgets
import ipywidgets as widgets
from IPython.display import display

queries = [
    "What are the key financial highlights and revenue growth drivers for NVIDIA in the most recent quarter?",
    "How does NVIDIA describe the impact of generative AI on its data center business and future outlook?",
    "Summarize the main advancements in NVIDIA’s GPU architecture",
    "What strategic partnerships or collaborations are highlighted, and how do they contribute to NVIDIA’s growth?",
    "How is NVIDIA addressing the challenges and opportunities in the automotive sector?",
    "What are the main components and benefits of the NVIDIA Omniverse platform?",
    "How does NVIDIA position its role in the AI ecosystem, and what are its competitive advantages?",
    "What are the key risks and uncertainties mentioned in the investor presentation?",
    "How does NVIDIA’s approach to sustainability and ESG (Environmental, Social, Governance) appear?",
    "What future product launches or technology roadmaps are discussed, and what is their expected impact on NVIDIA’s business?"
]
query_dropdown = widgets.Dropdown(options=queries, description='Select Query:', layout=widgets.Layout(width='90%'))
custom_query = widgets.Text(value='', placeholder='Or enter your own query', description='Custom:', layout=widgets.Layout(width='90%'))
output = widgets.Output()

def on_query_change(change):
    with output:
        output.clear_output()
        selected = custom_query.value if custom_query.value else query_dropdown.value
        print(f"{selected}")

query_dropdown.observe(on_query_change, names='value')
custom_query.observe(on_query_change, names='value')

In [27]:
display(query_dropdown, custom_query, output)

Dropdown(description='Select Query:', layout=Layout(width='90%'), options=('What are the key financial highlig…

Text(value='', description='Custom:', layout=Layout(width='90%'), placeholder='Or enter your own query')

Output()

Implement a cell for LLM-only answers

In [28]:
# LLM-only answer cell: answers the selected query using only the LLM, without retrieval.
from langchain_openai import ChatOpenAI

In [29]:
try:
    selected_query = custom_query.value if custom_query.value else query_dropdown.value
except Exception:
    selected_query = None

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
else:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
    prompt = f'''You are an expert analyst reviewing the NVIDIA Investor Presentation PDF. Answer the following question in detail, using evidence and insights that would be expected from such a document. If you are unsure, state your reasoning and what information might be missing.\n\nQuestion: {selected_query}'''
    response_llm = llm.invoke(prompt)
    print("LLM-only answer:")
    print(response_llm.content)

22:56:35 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
LLM-only answer:
NVIDIA's GPU architecture has seen significant advancements in recent years, as outlined in their Investor Presentation PDF. One of the key advancements highlighted is the introduction of the NVIDIA Ampere architecture. This architecture represents a major leap forward in terms of performance, efficiency, and scalability.

The Ampere architecture features several key innovations that have enabled NVIDIA to deliver industry-leading performance in a wide range of applications. One of the most notable advancements is the use of third-generation Tensor Cores, which are specialized processing units designed to accelerate AI and machine learning workloads. These Tensor Cores are capable of delivering up to 20 times the performance of the previous generation, making them ideal for demanding AI applications such as deep learning and neural network training.

Another key advanc

RAG-augmented answer using LangChain Runnable API (v0.1.x+/v1.x)

In [ ]:
#!pip install --upgrade pydantic

In [ ]:
#!pip install --upgrade langchain langchain-core langchain-openai

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage

In [31]:
try:
    selected_query = custom_query.value if custom_query.value else query_dropdown.value
except Exception:
    selected_query = None

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
else:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

    def format_docs(docs):
        return "\n\n".join([doc.page_content for doc in docs])

    def build_prompt(question, context):
        return f'''You are an expert analyst. Using only the context from the NVIDIA Investor Presentation PDF provided below, answer the following question in detail. Reference specific facts, figures, or statements from the context. If the context does not contain enough information, explain what is missing.\n\nContext:\n{context}\n\nQuestion: {question}'''

    rag_chain = RunnableMap({
        "context": retriever,
        "question": RunnablePassthrough()
    }) | RunnableLambda(lambda inputs: build_prompt(inputs["question"], format_docs(inputs["context"]))) | llm

    # prompt_template = ChatPromptTemplate.from_template(
    #         """
    #         You are an expert analyst. Using ONLY the context below from the NVIDIA Investor Presentation PDF, 
    #         answer the question in detail. If information is missing, say so.

    #         Context:
    #         {context}

    #         Question: {question}
    #         """
    #     )
    # rag_chain = (
    #         RunnableMap({
    #             "context": retriever | RunnableLambda(format_docs),
    #             "question": RunnablePassthrough()
    #         })
    #         | prompt_template
    #         | llm
    #     )

    result = rag_chain.invoke(selected_query)
    print("RAG-augmented answer:")
    print(result.content if hasattr(result, 'content') else result)

    #Show source chunks and page numbers
    docs = retriever.invoke(selected_query)
    for i, doc in enumerate(docs):
        page = doc.metadata.get('page', doc.metadata.get('page_number', 'N/A'))
        print(f"Chunk {i+1} (Page {page}):\n{doc.page_content[:400]}\n")

22:56:56 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
22:57:00 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
RAG-augmented answer:
NVIDIA's GPU architecture has made significant advancements in the field of accelerated computing. The company's platform is installed in hundreds of millions of computers, available in every cloud and from every server maker, and powers 76% of the TOP500 supercomputers. NVIDIA's GPU architecture has attracted 4.5 million developers and supports a rapidly growing universe of applications and industry innovation. The company's full-stack innovation across silicon, systems, and software includes a wide range of products and technologies such as cuNumeric, CV-CUDA, cuQuantum, Parabricks, Sionna, Jetpack, RAPIDS, Spark, cuDNN, cuGraph, TensorRT, Triton, Deepstream, Flare, DOCA, Mag, IO, Aerial, HPC, AI, Omniverse, RTX, DGX, HGX, EGX, OVX, AGXSuper, POD, IGX, DPUCPU, GPU, an

In [44]:
# Side-by-side comparison: LLM-only vs RAG-augmented response with refined prompts
from IPython.display import display, Markdown
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda

try:
    selected_query = custom_query.value if custom_query.value else query_dropdown.value
except Exception:
    selected_query = None

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
else:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

    # LLM-only response with improved prompt
    llm_prompt = f'''You are an expert analyst reviewing the NVIDIA Investor Presentation PDF. 
    Answer the following question in detail, using evidence and insights that would be expected 
    from such a document. 
    
    If you are unsure, state your reasoning and what information might be missing.
    \n\nQuestion: {selected_query}'''
    
    llm_only_response = llm.invoke(llm_prompt)
    llm_only_text     = llm_only_response.content if hasattr(llm_only_response, 'content') else llm_only_response

    # RAG-augmented response with improved prompt
    def format_docs(docs):
        return "\n\n".join([doc.page_content for doc in docs])
    
    def build_prompt(question, context):
        return f'''You are an expert analyst. 
    Using only the context from the NVIDIA Investor Presentation PDF provided below, 
    answer the following question in detail. 
    
    Reference specific facts, figures, or statements from the context. 
    
    If the context does not contain enough information, 
    
    explain what is missing.\n\nContext:\n{context}\n\nQuestion: {question}'''

    rag_chain = RunnableMap({
        "context": retriever,
        "question": RunnablePassthrough()
    }) | RunnableLambda(lambda inputs: build_prompt(inputs["question"], format_docs(inputs["context"]))) | llm
    
    rag_response = rag_chain.invoke(selected_query)
    rag_text     = rag_response.content if hasattr(rag_response, 'content') else rag_response

    # Display side-by-side comparison using Markdown
    display(Markdown(f"""
| LLM-only Response | RAG-augmented Response |
|-------------------|-----------------------|
| {llm_only_text.replace(chr(10), '<br>')} | {rag_text.replace(chr(10), '<br>')} |
"""))

12:41:51 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
12:41:52 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
12:41:55 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



| LLM-only Response | RAG-augmented Response |
|-------------------|-----------------------|
| NVIDIA's GPU architecture has seen significant advancements in recent years, as outlined in their Investor Presentation PDF. One of the key advancements highlighted is the introduction of the Ampere architecture, which represents a major leap forward in terms of performance and efficiency.<br><br>The Ampere architecture, as detailed in the presentation, features several key improvements over its predecessor, the Turing architecture. These include a significant increase in the number of CUDA cores, which are the processing units responsible for executing tasks on the GPU. This increase in CUDA cores results in a substantial boost in performance, making the Ampere architecture well-suited for demanding applications such as artificial intelligence, data analytics, and gaming.<br><br>Another important advancement in NVIDIA's GPU architecture is the integration of second-generation RT cores and third-generation Tensor cores. These specialized cores are designed to accelerate ray tracing and AI workloads, respectively, providing a significant performance improvement in these areas. Ray tracing, in particular, is a cutting-edge rendering technique that simulates the behavior of light in a scene, resulting in more realistic and immersive graphics.<br><br>Furthermore, NVIDIA has made significant strides in power efficiency with the Ampere architecture. By leveraging advanced manufacturing processes and architectural optimizations, the company has been able to deliver higher performance while consuming less power. This is crucial for applications that require high computational power but also need to operate within strict power constraints, such as data centers and mobile devices.<br><br>Overall, NVIDIA's GPU architecture advancements, as outlined in the Investor Presentation PDF, demonstrate the company's commitment to pushing the boundaries of performance, efficiency, and innovation in the GPU market. The Ampere architecture represents a significant step forward in terms of computational power, specialized workload acceleration, and power efficiency, positioning NVIDIA as a leader in the industry.<br><br>However, the presentation may lack specific performance metrics or benchmarks comparing the Ampere architecture to previous generations or competitors. Without this information, it may be challenging to fully assess the impact and significance of these advancements in NVIDIA's GPU architecture. | NVIDIA's GPU architecture has made significant advancements in accelerated computing, with a focus on full-stack innovation across silicon, systems, and software. The company's platform is installed in hundreds of millions of computers, available in every cloud and from every server maker, and powers 76% of the TOP500 supercomputers. NVIDIA's GPU architecture has attracted 4.5 million developers and supports a rapidly growing universe of applications and industry innovation. The company's full-stack platforms, including NVIDIA HPC, NVIDIA AI, and NVIDIA Omniverse, accelerate high-performance computing, AI, and industrial digitalization workloads. NVIDIA's GPU architecture also offers a wide range of acceleration libraries, AI application frameworks, and cloud-to-edge solutions. The company's ecosystem of developers can engage with NVIDIA through CUDA, libraries, pre-trained AI models, SDKs, and other development tools. Overall, NVIDIA's GPU architecture has shown advancements in accelerating software and scaling compute by a Million-X, going beyond Moore's law and demanding deep understanding of the problem domain. |


In [33]:
# Show the source chunks used by the RAG-augmented LLM call for the selected query, displaying the entire chunk with word wrap
from IPython.display import display, Markdown

try:
    selected_query = custom_query.value if custom_query.value else query_dropdown.value
except Exception:
    selected_query = None

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
else:
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
    docs = retriever.invoke(selected_query)
    for i, doc in enumerate(docs):
        page = doc.metadata.get('page', doc.metadata.get('page_number', 'N/A'))
        display(Markdown(f"""---\n**Chunk {i+1} (Page {page}):**\n\n<pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>{doc.page_content}</pre>\n"""))

22:58:38 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


---
**Chunk 1 (Page 22):**

<pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>NVIDIA Overview</pre>


---
**Chunk 2 (Page 23):**

<pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>24
Headquarters: Santa Clara, CA
NVIDIA pioneered accelerated computing to help solve impactful 
challenges classical computers cannot.  A quarter of a century in the 
making, NVIDIA accelerated computing is broadly recognized as the 
way to advance computing as Moore’s law ends and AI lifts off. 
NVIDIA’s platform is installed in several hundred million computers, 
is available in every cloud and from every server maker, powers 76% 
of the TOP500 supercomputers, and boasts 4.5 million developers.</pre>


---
**Chunk 3 (Page 24):**

<pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>NVIDIA’s Accelerated Computing Platform
Full-stack innovation across silicon, systems and software
cuNumeric CV-CUDA cuQuantum Parabricks Sionna Jetpack
RAPIDS Spark cuDNN cuGraph TensorRT Triton Deepstream Flare
DOCA Mag IO Aerial
NVIDIA HPC NVIDIA AI NVIDIA Omniverse
RTX DGX HGX EGX OVX AGXSuper
POD IGX
DPUCPUGPU
3-CHIPS
PLATFORMS
AI APPLICATION FRAMEWORK
ACCELERATION
LIBRARIES
CLOUD-TO-EDGE
DATACENTER-TO-
ROBOTIC SYSTEMS
With nearly three decades of singular focus, 
NVIDIA is expert at accelerating software 
and scaling compute by a 
Million-X, going well beyond Moore’s law 
Accelerated computing requires full-stack 
innovation — optimizing across every layer 
of computing — from silicon and systems to 
software and algorithms, demanding deep 
understanding of the problem domain
Our full-stack platforms — NVIDIA HPC, 
NVIDIA AI, and NVIDIA Omniverse — 
accelerate high performance computing, 
AI and industrial digitalization workloads
We accelerate workloads at data center</pre>


---
**Chunk 4 (Page 29):**

<pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>The NVIDIA accelerated computing platform has attracted 
the largest ecosystem of developers, supporting a rapidly 
growing universe of applications and industry innovation
Developers can engage with NVIDIA through CUDA — 
our parallel computing programming model introduced 
in 2006 — or at higher layers of the stack, including libraries, 
pre-trained AI models, SDKs and other development tools
NVIDIA’s Accelerated Computing Ecosystem
Developers
 CUDA Downloads*
AI Startups
 GPU-Accelerated 
Applications
2020 2023
6K
15K
2020 2023
4.5M
1.8M
2020 2023
48M
20M
2020 2023
3,200
700
300 Libraries
600 AI Models
100 Updated in the Last Year
*Cumulative</pre>


#### Structuring the output

In [34]:
# Define Pydantic schemas for structured output from both LLM-only and RAG-augmented calls
from pydantic import BaseModel, Field
from typing import List, Optional

class AnswerChunk(BaseModel):
    page_number: Optional[int]  = Field(None, description="Page number of the source chunk")
    content: str                = Field(...,  description="Text content of the chunk")

class StructuredAnswer(BaseModel):
    summary: str = Field(..., description="Concise summary of the answer")
    key_points: List[str]               = Field(..., description="List of key points")
    supporting_evidence: List[str]      = Field(..., description="Supporting facts, quotes, or references")
    missing_information: Optional[str]  = Field(None, description="What information is missing or uncertain")

class RAGStructuredAnswer(StructuredAnswer):
    source_chunks: Optional[List[AnswerChunk]] = Field(None, description="Chunks used for RAG-augmented answer")

# Example usage:
# answer = StructuredAnswer(summary="...", key_points=["..."], supporting_evidence=["..."], missing_information="...")

#### Updated Prompt Templates for Structured Output
The following prompt templates are designed to guide the LLM to return answers in the structured format defined by the Pydantic schemas above.

In [35]:
# LLM-only structured prompt template
llm_structured_prompt = '''
You are an expert analyst reviewing the NVIDIA Investor Presentation PDF. Answer the following question in detail, and structure your response as a JSON object with the following fields:
- summary: A concise summary of the answer.
- key_points: A list of key points addressing the question.
- supporting_evidence: A list of supporting facts, quotes, or references.
- missing_information: What information is missing or uncertain (if any).

Return only a valid JSON object matching this schema.

Question: {selected_query}
'''

In [36]:
# RAG-augmented structured prompt template
rag_structured_prompt = '''
You are an expert analyst. Using only the context from the NVIDIA Investor Presentation PDF provided below, answer the following question in detail, and structure your response as a JSON object with the following fields:
- summary: A concise summary of the answer.
- key_points: A list of key points addressing the question.
- supporting_evidence: A list of supporting facts, quotes, or references from the context.
- missing_information: What information is missing or uncertain (if any).
- source_chunks: For each chunk used, include its page number and content.

Return only a valid JSON object matching this schema.

Context:
{context}

Question: {selected_query}
'''

#### Side-by-side comparison: LLM-only vs RAG-augmented response with refined prompts

In [37]:
# Side-by-side comparison: LLM-only vs RAG-augmented response using structured prompts and outputs
from IPython.display import display, Markdown
import json
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda

In [40]:
# Side-by-side comparison: LLM-only vs RAG-augmented response using pre-defined structured prompts and outputs
from IPython.display import display, Markdown
import json
from langchain_openai import ChatOpenAI

try:
    selected_query = custom_query.value if custom_query.value else query_dropdown.value
except Exception:
    selected_query = None

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
else:
    # Use GPT-3.5 for LLM-only, GPT-4 for RAG-augmented
    llm_only      = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
    llm_rag       = ChatOpenAI(model="gpt-4", temperature=0.2)
    
    retriever     = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

    # LLM-only structured response
    llm_only_response = llm_only.invoke(llm_structured_prompt.format(selected_query=selected_query))
    
    try:
        llm_structured = json.loads(llm_only_response.content if hasattr(llm_only_response, 'content') else llm_only_response)
    except Exception as e:
        llm_structured = {'summary': 'Error parsing LLM output', 'key_points': [], 'supporting_evidence': [], 'missing_information': str(e)}

    # RAG-augmented structured response
    docs    = retriever.invoke(selected_query)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    rag_prompt = rag_structured_prompt.format(context=context, selected_query=selected_query)
    
    rag_response = llm_rag.invoke(rag_prompt)
    
    def get_chunks(docs):
        return [{'page_number': doc.metadata.get('page', doc.metadata.get('page_number', None)), 'content': doc.page_content} for doc in docs]
    
    try:
        rag_structured = json.loads(rag_response.content if hasattr(rag_response, 'content') else rag_response)
    except Exception as e:
        rag_structured = {'summary': 'Error parsing RAG output', 'key_points': [], 'supporting_evidence': [], 'missing_information': str(e), 'source_chunks': get_chunks(docs)}

10:11:27 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
10:11:28 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
10:11:50 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [41]:
# Display side-by-side comparison using Markdown
def format_structured_output(struct):
    md = f"**Summary:** {struct.get('summary', '')}\n\n"
    md += "**Key Points:**\n" + "\n".join([f"- {kp}" for kp in struct.get('key_points', [])]) + "\n\n"
    md += "**Supporting Evidence:**\n" + "\n".join([f"- {ev}" for ev in struct.get('supporting_evidence', [])]) + "\n\n"
    md += f"**Missing Information:** {struct.get('missing_information', '')}\n\n"
    if 'source_chunks' in struct and struct['source_chunks'] is not None:
        md += "**Source Chunks:**\n"
        for i, chunk in enumerate(struct['source_chunks']):
            md += f"- Chunk {i+1} (Page {chunk.get('page_number', 'N/A')}):\n<pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>{chunk.get('content', '')}</pre>\n"
    return md

In [42]:
display(Markdown(f"""
| LLM-only Structured Response | RAG-augmented Structured Response |
|-----------------------------|-------------------------------|
| {format_structured_output(llm_structured).replace(chr(10), '<br>')} | {format_structured_output(rag_structured).replace(chr(10), '<br>')} |
"""))


| LLM-only Structured Response | RAG-augmented Structured Response |
|-----------------------------|-------------------------------|
| **Summary:** NVIDIA's GPU architecture advancements include increased performance, efficiency, and AI capabilities.<br><br>**Key Points:**<br>- Introduction of Ampere architecture for improved performance and efficiency<br>- Enhanced ray tracing capabilities with RTX technology<br>- Integration of AI capabilities with Tensor Cores for deep learning tasks<br><br>**Supporting Evidence:**<br>- NVIDIA introduced the Ampere architecture in their Investor Presentation PDF, highlighting its performance and efficiency improvements.<br>- The RTX technology in NVIDIA GPUs enables advanced ray tracing capabilities for realistic lighting and reflections in gaming and professional applications.<br>- The integration of Tensor Cores in NVIDIA GPUs allows for accelerated AI tasks such as deep learning and neural network training.<br><br>**Missing Information:** <br><br> | **Summary:** The main advancements in NVIDIA's GPU architecture are not explicitly mentioned in the provided context. However, it is clear that NVIDIA has pioneered accelerated computing, which is recognized as the way to advance computing as Moore’s law ends and AI lifts off. NVIDIA's platform is installed in several hundred million computers, powers 76% of the TOP500 supercomputers, and boasts 4.5 million developers. The company has a full-stack innovation across silicon, systems, and software, and its platforms accelerate high performance computing, AI, and industrial digitalization workloads.<br><br>**Key Points:**<br>- NVIDIA pioneered accelerated computing.<br>- NVIDIA's platform is installed in several hundred million computers.<br>- NVIDIA powers 76% of the TOP500 supercomputers.<br>- NVIDIA has 4.5 million developers.<br>- NVIDIA has full-stack innovation across silicon, systems, and software.<br>- NVIDIA's platforms accelerate high performance computing, AI, and industrial digitalization workloads.<br><br>**Supporting Evidence:**<br>- NVIDIA pioneered accelerated computing to help solve impactful challenges classical computers cannot.<br>- NVIDIA’s platform is installed in several hundred million computers, is available in every cloud and from every server maker, powers 76% of the TOP500 supercomputers, and boasts 4.5 million developers.<br>- With nearly three decades of singular focus, NVIDIA is expert at accelerating software and scaling compute by a Million-X, going well beyond Moore’s law.<br>- Accelerated computing requires full-stack innovation — optimizing across every layer of computing — from silicon and systems to software and algorithms, demanding deep understanding of the problem domain.<br>- Our full-stack platforms — NVIDIA HPC, NVIDIA AI, and NVIDIA Omniverse — accelerate high performance computing, AI and industrial digitalization workloads.<br><br>**Missing Information:** The specific advancements in NVIDIA's GPU architecture are not mentioned in the provided context.<br><br>**Source Chunks:**<br>- Chunk 1 (Page 24):<br><pre style='white-space:pre-wrap;word-wrap:break-word;background:#f8f8f8;padding:8px;border-radius:4px'>NVIDIA Overview<br><br>24<br>Headquarters: Santa Clara, CA<br>NVIDIA pioneered accelerated computing to help solve impactful <br>challenges classical computers cannot.  A quarter of a century in the <br>making, NVIDIA accelerated computing is broadly recognized as the <br>way to advance computing as Moore’s law ends and AI lifts off. <br>NVIDIA’s platform is installed in several hundred million computers, <br>is available in every cloud and from every server maker, powers 76% <br>of the TOP500 supercomputers, and boasts 4.5 million developers.<br><br>NVIDIA’s Accelerated Computing Platform<br>Full-stack innovation across silicon, systems and software<br>cuNumeric CV-CUDA cuQuantum Parabricks Sionna Jetpack<br>RAPIDS Spark cuDNN cuGraph TensorRT Triton Deepstream Flare<br>DOCA Mag IO Aerial<br>NVIDIA HPC NVIDIA AI NVIDIA Omniverse<br>RTX DGX HGX EGX OVX AGXSuper<br>POD IGX<br>DPUCPUGPU<br>3-CHIPS<br>PLATFORMS<br>AI APPLICATION FRAMEWORK<br>ACCELERATION<br>LIBRARIES<br>CLOUD-TO-EDGE<br>DATACENTER-TO-<br>ROBOTIC SYSTEMS<br>With nearly three decades of singular focus, <br>NVIDIA is expert at accelerating software <br>and scaling compute by a <br>Million-X, going well beyond Moore’s law <br>Accelerated computing requires full-stack <br>innovation — optimizing across every layer <br>of computing — from silicon and systems to <br>software and algorithms, demanding deep <br>understanding of the problem domain<br>Our full-stack platforms — NVIDIA HPC, <br>NVIDIA AI, and NVIDIA Omniverse — <br>accelerate high performance computing, <br>AI and industrial digitalization workloads<br>We accelerate workloads at data center</pre><br> |


#### Validating the veracity of LLM and RAG responses

**1. Self-Assessment Prompting for LLM Veracity**

Self-Assessment Prompting is a technique to improve transparency and trust in LLM-generated answers. By modifying the prompt, you instruct the LLM to reflect on its own output and provide a confidence rating for each part of the answer.

**How it works:**
- For each key point or claim, the LLM specifies:
    - Whether it is a “Known Fact” (directly supported by evidence or well-established knowledge) or “Likely Information” (inferred or assumed).
    - A confidence score (e.g., High, Medium, Low).
    - A brief explanation for the rating.

**Example prompt addition:**
For each key point in your answer, specify:
- Is this a “Known Fact” or “Likely Information”?
- Rate your confidence (High/Medium/Low).
- Briefly explain your reasoning.

This approach helps users quickly identify which parts of the answer are reliable and which may require further verification or manual review.

`Update the Schema`

Add a new class for key point assessment:

In [ ]:
class KeyPointAssessment(BaseModel):
    key_point: str
    fact_type: str   # "Known Fact" or "Likely Information"
    confidence: str  # "High", "Medium", "Low"
    explanation: str

class StructuredAnswerWithAssessment(BaseModel):
    summary: str
    key_points: List[KeyPointAssessment]
    supporting_evidence: List[str]
    missing_information: Optional[str]

In [ ]:
# LLM-only structured prompt template with key point assessment
llm_structured_prompt = '''
You are an expert analyst reviewing the NVIDIA Investor Presentation PDF.
Answer the following question in detail, and structure your response as a JSON object with the following fields:
- summary: A concise summary of the answer.
- key_points: For each key point, include:
    - key_point: The key point itself.
    - fact_type: Is this a "Known Fact" (directly supported by evidence or well-established knowledge) or "Likely Information" (inferred or assumed)?
    - confidence: Rate your confidence (High/Medium/Low).
    - explanation: Briefly explain your reasoning for the rating.
    - supporting_evidence: A list of supporting facts, quotes, or references for this key point.

Do NOT include a top-level 'supporting_evidence' or 'missing_information' field.

Return only a valid JSON object matching this schema:
{{
  "summary": "...",
  "key_points": [
    {{
      "key_point": "...",
      "fact_type": "...",
      "confidence": "...",
      "explanation": "...",
      "supporting_evidence": ["..."]
    }}
  ]
}}

Question: {selected_query}
'''

In [ ]:
# Run LLM call with refined prompt and load response into revamped Pydantic structure
from langchain_openai import ChatOpenAI
import json
from pydantic import BaseModel, Field, ValidationError
from typing import List
try:
    selected_query = custom_query.value if custom_query.value else query_dropdown.value
except Exception:
    selected_query = None

class KeyPointAssessment(BaseModel):
    key_point: str
    fact_type: str   # "Known Fact" or "Likely Information"
    confidence: str  # "High", "Medium", "Low"
    explanation: str
    supporting_evidence: List[str]

class StructuredAnswerWithAssessment(BaseModel):
    summary: str
    key_points: List[KeyPointAssessment]

def safe_format(template, **kwargs):
    # Only replace the intended {selected_query} variable, leave double curly braces untouched
    return template.replace('{selected_query}', kwargs.get('selected_query', ''))

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
else:
    #llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)
    prompt = safe_format(llm_structured_prompt, selected_query=selected_query)
    response = llm_only.invoke(prompt)
    try:
        raw_json = response.content if hasattr(response, 'content') else response
        parsed = json.loads(raw_json)
        answer = StructuredAnswerWithAssessment(**parsed)
        print("StructuredAnswerWithAssessment loaded successfully.")
        print(answer.model_dump_json(indent=2))
    except (json.JSONDecodeError, ValidationError, TypeError) as e:
        print("Error parsing or validating LLM output:", e)

In [ ]:
# Pretty-print selected query and structured answer using Markdown
from IPython.display import display, Markdown

def format_key_point(kp):
    return (
        f"**Key Point:** {kp.key_point}\n"
        f"- Fact Type: {kp.fact_type}\n"
        f"- Confidence: {kp.confidence}\n"
        f"- Explanation: {kp.explanation}\n"
        f"- Supporting Evidence: " + ", ".join(kp.supporting_evidence)
    )

if not selected_query or not selected_query.strip():
    print("Please select or enter a query above.")
elif 'answer' not in locals():
    print("No structured answer available. Please run the previous cell.")
else:
    md = f"""
### Selected Query

{selected_query}

### Structured LLM Answer (Self-Assessed)

**Summary:** {answer.summary}\n

**Key Points:**\n
"""
    for kp in answer.key_points:
        md += format_key_point(kp) + "\n\n"

    display(Markdown(md))

#### Extract slide summaries as text using GPT-4V:

GPT-4V (Vision) can process images (e.g., PDF slides) and generate accurate text summaries. 

This step is supported via the OpenAI API or LangChain’s vision modules.

In [ ]:
# Extract slide summaries as text using GPT-4V (Vision)
from openai import OpenAI
import base64
from PIL import Image
import io
import os

In [ ]:
# Path to PDF and output directory for slide images
pdf_path      = r"D:\Makesh\Working\AI\RPS\Assignments\Capstone\cap-02\nvda-f3q24-investor-presentation-final.pdf"
slide_img_dir = "slide_images"

os.makedirs(slide_img_dir, exist_ok=True)

In [ ]:
# Convert PDF slides to images (one image per page)
# time taken = 15+ mins
from pdf2image import convert_from_path

slides = convert_from_path(pdf_path)

In [ ]:
len(slides)

In [ ]:
# takes a while to run: saves each slide as PNG image
# Save slide images to disk
# time taken = 5+ mins 
slide_image_paths = []

for i, slide in enumerate(slides):
    img_path = os.path.join(slide_img_dir, f"slide_{i+1}.png")
    slide.save(img_path, "PNG")
    slide_image_paths.append(img_path)

#### GPT-4V (gpt-4-vision-preview) Cost and Time Estimate for 65 Images

**Time per image:**
- Each API call typically takes 10–30 seconds, depending on image size and network latency.
- For 65 images, expect 10–30 minutes total if processed sequentially.
- Parallelization (batching) can reduce total time, but may be limited by API rate limits.

**Cost per image:**
- As of late 2025, OpenAI’s pricing for gpt-4-vision-preview is typically:
  - $0.01 per image (input) for images up to 1MB.
  - $0.03 per 1,000 output tokens (text).
- For a short summary (≤512 tokens), each image will cost about $0.01–$0.02.

**Total cost estimate:**
- 65 images × $0.01/image ≈ $0.65 (image input)
- 65 images × (≤512 tokens) × $0.03/1,000 tokens ≈ $1.00 (output text)
- **Total: $1.50–$2.00 USD** for 65 slides.

**Summary:**
- **Time:** 10–30 minutes (sequential), faster if batched.
- **Cost:** $1.50–$2.00 USD for 65 images (assuming standard OpenAI pricing).

In [43]:
openai_client = OpenAI()

NameError: name 'OpenAI' is not defined

In [ ]:
# Test GPT-4V slide summary on a random slide image
import random

# Pick a random slide image
if slide_image_paths:
    test_img_path = random.choice(slide_image_paths)
    
    print(f"Testing with slide: {test_img_path}")
    
    with open(test_img_path, "rb") as img_file:
        img_bytes = img_file.read()
        img_b64   = base64.b64encode(img_bytes).decode("utf-8")
    
    prompt = "Summarize the key points and content of this investor presentation slide."
    
    response = openai_client.chat.completions.create(
        model="gpt‑4o‑mini",
        messages=[
            {"role": "system", "content": "You are an expert analyst for investor presentations."},
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ],
        max_tokens=512
    )
    summary = response.choices[0].message.content
    print("Slide Summary:")
    print(summary)
else:
    print("No slide images found. Please run the slide extraction cells first.")

In [ ]:
# Use GPT-4V to summarize each slide image

slide_summaries = []

for img_path in slide_image_paths:
    with open(img_path, "rb") as img_file:
        img_bytes = img_file.read()
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    
    # GPT-4V vision prompt
    # 
    prompt = "Summarize the key points and content of this investor presentation slide."
    response = openai_client.chat.completions.create(
        model="gpt-4-vision-preview",
        messages=[
            {"role": "system", "content": "You are an expert analyst for investor presentations."},
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ],
        max_tokens=512
    )
    summary = response.choices[0].message.content
    slide_summaries.append({"slide": img_path, "summary": summary})

In [ ]:
# Display the first few slide summaries
for i, item in enumerate(slide_summaries[:3]):
    print(f"Slide {i+1}: {item['slide']}")
    print(item['summary'])
    print("-"*40)

In [ ]:
# Test GPT-4V slide summary on a random slide image (updated for current model)
import random

# Pick a random slide image
if slide_image_paths:
    test_img_path = random.choice(slide_image_paths)
    print(f"Testing with slide: {test_img_path}")
    with open(test_img_path, "rb") as img_file:
        img_bytes = img_file.read()
        img_b64   = base64.b64encode(img_bytes).decode("utf-8")
        
    prompt = "Summarize the key points and content of this investor presentation slide."
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",  # Use the current vision-enabled model
        messages=[
            {"role": "system", "content": "You are an expert analyst for investor presentations."},
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ],
        max_tokens=512
    )
    summary = response.choices[0].message.content
    print("Slide Summary:")
    print(summary)
else:
    print("No slide images found. Please run the slide extraction cells first.")

In [ ]:
# Display the random slide image used for the summary above
from IPython.display import display, Image as IPyImage

if slide_image_paths:
    display(IPyImage(filename=test_img_path))
else:
    print("No slide images found. Please run the slide extraction cells first.")

#### strctured output from GPT-4V slide summaries

In [ ]:
# --- Updated Pydantic schema for slide extraction with CSV tables ---
from pydantic import BaseModel, Field
from typing import List, Optional

class TableCell(BaseModel):
    text: str = Field(..., description="Text content of the cell")
    is_header: Optional[bool] = Field(False, description="Is this cell a header?")

class TableRow(BaseModel):
    cells: List[TableCell] = Field(..., description="Cells in the row")

class Table(BaseModel):
    caption: Optional[str] = Field(None, description="Table caption or title")
    rows: List[TableRow] = Field(..., description="Rows of the table")
    csv: Optional[str] = Field(None, description="Table as comma-separated values (CSV) string")

class MathContent(BaseModel):
    latex: str = Field(..., description="Mathematical expression in LaTeX")
    description: Optional[str] = Field(None, description="Description or context of the formula")

class VisualElement(BaseModel):
    type: str = Field(..., description="Type of visual (chart, image, icon, etc.)")
    description: str = Field(..., description="Description of the visual element")
    text: Optional[str] = Field(None, description="Any text found within the visual")

class SlideContent(BaseModel):
    heading: Optional[str] = Field(None, description="Main title of the slide")
    subheadings: List[str] = Field(default_factory=list, description="Section titles or subtitles")
    paragraphs: List[str] = Field(default_factory=list, description="Main body text")
    bullet_points: List[str] = Field(default_factory=list, description="Bullet points")
    tables: List[Table] = Field(default_factory=list, description="Tables in the slide (each with optional CSV representation)")
    math: List[MathContent] = Field(default_factory=list, description="Mathematical content (formulas, equations)")
    visuals: List[VisualElement] = Field(default_factory=list, description="Charts, images, or other visuals")
    captions: List[str] = Field(default_factory=list, description="Captions for figures or tables")
    footnotes: List[str] = Field(default_factory=list, description="Footnotes or disclaimers")
    other_elements: List[str] = Field(default_factory=list, description="Any other relevant content")
    summary: Optional[str] = Field(None, description="One-sentence summary of the slide’s main message")

In [ ]:
# --- Extract, parse, and display a random slide using the robust Pydantic schema ---
import random
import base64
from openai import OpenAI
from pydantic import ValidationError
from IPython.display import display, Image as IPyImage


In [ ]:
openai_client = OpenAI()

if slide_image_paths:
    # Pick a random slide image
    test_img_path = random.choice(slide_image_paths)
    print(f"Selected slide: {test_img_path}")
    with open(test_img_path, "rb") as img_file:
        img_bytes = img_file.read()
        img_b64   = base64.b64encode(img_bytes).decode("utf-8")

    # Vision prompt for structured extraction
    vision_prompt = (
        "Extract all structured content from this investor presentation slide. "
        "Return a JSON object matching the following schema: "
        "{"
        "  heading: Main title of the slide (if any),"
        "  subheadings: List of section titles or subtitles,"
        "  paragraphs: List of main body text,"
        "  bullet_points: List of bullet points,"
        "  tables: List of tables, each with caption, rows (list of rows, each a list of cell texts), and csv (CSV string),"
        "  math: List of mathematical content (LaTeX and description),"
        "  visuals: List of visual elements (type, description, text),"
        "  captions: List of captions for figures or tables,"
        "  footnotes: List of footnotes or disclaimers,"
        "  other_elements: List of any other relevant content,"
        "  summary: One-sentence summary of the slide’s main message"
        "}"
        "Return only a valid JSON object."
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert analyst for investor presentations."},
            {"role": "user", "content": [
                {"type": "text", "text": vision_prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ],
        max_tokens=1024
    )
    content = response.choices[0].message.content
    #print("\nRaw model output:\n", content)
else:
    print("No slide images found. Please run the slide extraction cells first.")

In [ ]:
# --- Robust JSON extraction and parsing for model output ---
import re
from pydantic import ValidationError
from IPython.display import display, Markdown

# Assume 'content' contains the raw model output from the previous cell

def extract_json_from_text(text):
    """
    Extract the first JSON object or array from a string, even if extra text is present.
    """
    # Try to find the first curly brace block
    match = re.search(r'({[\s\S]*})', text)
    if match:
        return match.group(1)
    # Try to find the first bracket block (for arrays)
    match = re.search(r'(\[[\s\S]*\])', text)
    if match:
        return match.group(1)
    return None

if 'content' in locals():
    print("\nRaw model output (truncated to 1000 chars):\n", content[:1000])
    json_str = extract_json_from_text(content)
    if json_str:
        import json
        try:
            parsed = json.loads(json_str)
            slide_struct = SlideContent(**parsed)
            print("\nParsed SlideContent:")
            print(slide_struct.model_dump_json(indent=2))
            # Optionally, display as Markdown for readability
            md = f"**Heading:** {slide_struct.heading}\n\n"
            md += f"**Summary:** {slide_struct.summary}\n\n"
            md += f"**Paragraphs:** {slide_struct.paragraphs}\n\n"
            md += f"**Bullet Points:** {slide_struct.bullet_points}\n\n"
            md += f"**Visuals:** {slide_struct.visuals}\n\n"
            display(Markdown(md))
        except (json.JSONDecodeError, ValidationError, TypeError) as e:
            print("\nError parsing or validating extracted JSON:", e)
            print("Extracted JSON string:\n", json_str)
    else:
        print("\nNo JSON object found in model output. Please review the raw output above.")
else:
    print("No model output ('content') found. Run the previous cell first.")

#### Embed text summaries using OpenAI’s embedding models
Use OpenAI's embedding models to convert the text summaries extracted from the slides into vector representations. This will facilitate efficient retrieval during the RAG process.

#### How Are Text Summaries Prepared?

Text summaries for each slide are generated as follows:

1. **Slide Image Analysis:** Each slide image is sent to the OpenAI Vision model (e.g., GPT-4 Vision or GPT-4o-mini) with a prompt requesting structured extraction of all key content.
2. **Model Extraction:** The model analyzes all visible elements on the slide, including text, visuals, and layout, and returns a structured JSON response.
3. **Summary Field:** The `summary` field in the resulting Pydantic `SlideContent` object is a concise, one-sentence summary of the slide’s main message.
4. **Content Basis:** This summary is not generic—it is generated by the model to capture the most important point or theme of the slide, based on all available information.

**In short:**
> The text summaries are prepared by GPT-4 Vision, which reviews the entire slide (text, visuals, layout) and generates a concise summary reflecting the slide’s main message.

#### create redis vector index


In [ ]:
# --- Redis index and storage structure for advanced RAG ---
# This cell creates a RediSearch index with fields matching all extracted slide fields, including image storage
# and prepares each slide hash to store both structured content and the image (as base64 or path)

from redis.commands.search.field import TextField, TagField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType
import base64
import os

r = redis.Redis.from_url(REDIS_URL)

# Define index fields to match SlideContent schema and image
fields = [
    TextField("heading"),
    TextField("summary"),
    TextField("paragraphs"),
    TextField("bullet_points"),
    TextField("tables"),
    TextField("math"),
    TextField("visuals"),
    TextField("captions"),
    TextField("footnotes"),
    TextField("other_elements"),
    TextField("content"),  # Full JSON
    TextField("image_path"),  # Path to slide image
    TextField("image_b64"),   # Optionally, base64-encoded image
    VectorField("embedding", "FLAT", {"TYPE": "FLOAT32", "DIM": 1536, "DISTANCE_METRIC": "COSINE"})
]

# Create the index (if not exists)
try:
    r.ft("slide_index").create_index(fields, definition=IndexDefinition(prefix=["slide:"], index_type=IndexType.HASH))
    print("Redis index 'slide_index' created.")
except Exception as e:
    print("Index may already exist or error:", e)

# Example: Store slide content and image
for i, (slide, embedding) in enumerate(zip(slide_structs, embeddings)):
    slide_key = f"slide:{i+1}"
    img_path = slide_image_paths[i] if i < len(slide_image_paths) else ""
    img_b64 = ""
    if os.path.exists(img_path):
        with open(img_path, "rb") as img_file:
            img_b64 = base64.b64encode(img_file.read()).decode("utf-8")
    r.hset(slide_key, mapping={
        "heading": slide.heading or "",
        "summary": slide.summary or "",
        "paragraphs": json.dumps(slide.paragraphs),
        "bullet_points": json.dumps(slide.bullet_points),
        "tables": json.dumps([t.model_dump() for t in slide.tables]),
        "math": json.dumps([m.model_dump() for m in slide.math]),
        "visuals": json.dumps([v.model_dump() for v in slide.visuals]),
        "captions": json.dumps(slide.captions),
        "footnotes": json.dumps(slide.footnotes),
        "other_elements": json.dumps(slide.other_elements),
        "content": slide.model_dump_json(),
        "image_path": img_path,
        "image_b64": img_b64,
        "embedding": json.dumps(embedding),
    })
    print(f"Indexed {slide_key} with image and content in Redis.")

# This structure supports high-end/complex RAG: you can search, retrieve, and cross-reference all slide content and images.

#### Hybrid RAG Workflow: Step-by-Step

1. **Chunked PDF Extraction (Broad Context)**
   - Extract and chunk text from each PDF page, keeping track of page numbers.
   - Store each chunk in Redis with metadata: `chunk_id`, `page_number`, and embedding.
   - Enables broad, context-rich retrieval for open-ended or cross-slide queries.

2. **Slide-Specific Extraction (Fine-Grained, Structured)**
   - For each slide, use vision models to extract structured fields: heading, bullet points, tables, images, plots, etc.
   - Store each slide’s structured content and image (as path or base64) in Redis, indexed by `slide:{n}`.
   - Enables precise, element-level retrieval for targeted queries.

3. **Unified Redis Index**
   - Create a RediSearch index with fields for both chunked and structured data:
     - For chunks: `chunk_id`, `page_number`, `chunk_text`, `embedding`
     - For slides: `slide_id`, `heading`, `summary`, `tables`, `visuals`, `image_path`, `image_b64`, `embedding`, etc.
   - Use a common field (e.g., `page_number` or `slide_id`) to link chunked and structured data.

4. **Hybrid Retrieval Workflow**
   - For a user query:
     - Retrieve top-k relevant text chunks (broad context) using semantic search.
     - Retrieve structured slide(s) and elements (precise facts, visuals) by page/slide number or metadata.
     - Merge both sources and pass to the LLM for answer synthesis.


- Extract slide summaries as text using GPT4-V
- Embed text summaries using OpenAI’s embedding models
- Index text summary embeddings in Redis hashes, referenced by a primary key
- Encode raw images as base64 strings and store them in Redis hashes with a primary key

For this use case, LangChain provides the MultiVector Retriever to index documents and summaries efficiently. The benefit of this approach is that we can employ commonly used text embeddings to index image summaries just like any other text, avoiding the need for more specialized and less mature multimodal embeddings.